
# Exercises XP : Evaluating LLMs for Summarization



## What you will learn
- Hands-on evaluation for summarization: accuracy vs. ROUGE.
- Strengths/weaknesses of metrics and model size comparisons.
- Using Hugging Face `transformers` + `evaluate` for quick experiments.
- Data loading, sampling, preprocessing, and debugging model outputs.

**Create**: evaluation scripts, comparison tables, custom metrics, and short analyses.


In [6]:

# Part I. Setup (run once per runtime)
# Install minimal deps; keep quiet to reduce noise.
!pip -q install rouge_score==0.1.2 evaluate datasets transformers accelerate nltk --quiet

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True


### Part II. Dataset loading and exploration
Preferred dataset: [abisee/cnn_dailymail](https://huggingface.co/datasets/abisee/cnn_dailymail) (map `article` -> `prompt_text`, `highlights` -> `prompt_title`).
- If you have local train/test CSVs with `prompt_text` / `prompt_title`, set the paths below.
- Otherwise, we will auto-sample a small slice from the HF dataset to keep things light.
- Show a couple of rows for a sanity check.
If HF download fails, a tiny fallback sample is used.


In [1]:
from huggingface_hub import login
from google.colab import userdata

# Get your token from Colab secrets
hf_token = userdata.get('HUGGINGFACE_KEY') # Assuming you named your secret 'HUGGINGFACE_KEY'

# Login to Hugging Face Hub
login(token=hf_token)

print("Successfully logged in to Hugging Face Hub.")

Successfully logged in to Hugging Face Hub.


In [2]:

import pandas as pd
from datasets import load_dataset

# Point to your data; leave empty to use the HF cnn_dailymail sample or fallback
train_path = ''  # e.g., '/content/train.csv'
test_path = ''   # e.g., '/content/test.csv'

fallback = pd.DataFrame([
    {
        'prompt_text': 'The cat sat on the mat and purred loudly while the sun set.',
        'prompt_title': 'Cat rests on mat at sunset'
    },
    {
        'prompt_text': 'Scientists discovered water on the moon, opening new research paths.',
        'prompt_title': 'Water found on the moon'
    },
    {
        'prompt_text': 'The local team won the championship after a dramatic final match.',
        'prompt_title': 'Local team clinches title'
    },
])

def load_and_sample(path, split_name, n):
    if path:
        df = pd.read_csv(path)
    else:
        try:
            hf_split = f"{split_name}[:{max(n, 3)}]"
            ds = load_dataset('abisee/cnn_dailymail', '3.0.0', split=hf_split)
            df = ds.to_pandas()[['article', 'highlights']].rename(columns={'article': 'prompt_text', 'highlights': 'prompt_title'})
        except Exception as exc:
            print(f"HF load failed ({exc}); using tiny fallback sample.")
            df = fallback.copy()
    return df.sample(min(n, len(df)), random_state=42).reset_index(drop=True)

train_df = load_and_sample(train_path, 'train', 100)
test_df = load_and_sample(test_path, 'test', 50)

display(train_df.head(2))


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

,prompt_text,prompt_title
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...



### Part III. Summarization with T5 (implement)
Tasks:
- Write `batch_generator` to yield mini-batches.
- Write `summarize_with_t5` using `t5-small` (or swap sizes) with GPU if available.
- Prefix inputs with "summarize: " and decode with `skip_special_tokens=True`.
- Clear CUDA cache between batches (`torch.cuda.empty_cache()`) and gc.collect().


In [3]:
import torch, gc
from transformers import AutoTokenizer, T5ForConditionalGeneration
from typing import Iterable, List

def batch_generator(items: List[str], batch_size: int):
    # TODO: yield slices of items of length batch_size
    for i in range(0, len(items), batch_size):
        yield items[i : i + batch_size]

def summarize_with_t5(texts: List[str], model_name: str = 't5-small', batch_size: int = 4, max_new_tokens: int = 32):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Load model with appropriate precision to save memory, if supported and on GPU
    if torch.cuda.is_available():
        if torch.cuda.is_bf16_supported():
            print(f"Loading {model_name} with bfloat16 precision on GPU.")
            model = T5ForConditionalGeneration.from_pretrained(model_name, torch_dtype=torch.bfloat16).to(device)
        else:
            # bfloat16 not supported, try float16 if CUDA is available
            print(f"bfloat16 not supported. Attempting to load {model_name} with float16 precision on GPU.")
            try:
                model = T5ForConditionalGeneration.from_pretrained(model_name, torch_dtype=torch.float16).to(device)
            except RuntimeError as e:
                print(f"Failed to load {model_name} with float16 on GPU ({e}). Falling back to CPU.")
                device = torch.device("cpu") # Force CPU
                model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)
    else:
        print(f"CUDA not available. Loading {model_name} on CPU.")
        model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

    all_summaries = []
    for i, batch_texts in enumerate(batch_generator(texts, batch_size)):
        # TODO: tokenize with prefix, generate, decode
        inputs = ["summarize: " + text for text in batch_texts]
        tokenized_inputs = tokenizer(inputs, return_tensors='pt', padding=True, truncation=True).to(device)

        summary_ids = model.generate(
            tokenized_inputs.input_ids,
            attention_mask=tokenized_inputs.attention_mask,
            max_new_tokens=max_new_tokens,
            num_beams=4, # Using beam search for potentially better quality
            early_stopping=True
        )

        decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True) for s in summary_ids]
        all_summaries.extend(decoded_summaries)

        # TODO: clear caches between batches
        if model.device.type == 'cuda': # Only clear CUDA cache if the model is actually on CUDA
            torch.cuda.empty_cache()
        gc.collect()

    return all_summaries

# RUN_FLAG keeps heavy generation optional for quick debugging
RUN_T5 = False
if RUN_T5:
    train_summaries_t5 = summarize_with_t5(train_df['prompt_text'].tolist(), model_name='t5-small', batch_size=2)
    display(pd.DataFrame({
        'prompt_text': train_df['prompt_text'],
        'reference_summary': train_df['prompt_title'],
        't5_small_summary': train_summaries_t5
    }).head())
else:
    print("Skipping T5 generation for speed. Set RUN_T5=True to execute.")

Skipping T5 generation for speed. Set RUN_T5=True to execute.



### Part IV. Accuracy evaluation (toy, likely near zero)
Implement a naive accuracy that checks exact string match between generated and reference summaries.
Discuss why this is harsh for free-form text (almost always zero).


In [4]:

from typing import List

def compute_accuracy(preds: List[str], refs: List[str]) -> float:
    matches = sum(1 for p, r in zip(preds, refs) if p.strip() == r.strip())
    return matches / max(len(refs), 1)

if 'train_summaries_t5' in locals():
    acc = compute_accuracy(train_summaries_t5, train_df['prompt_title'].tolist())
    print(f"Exact-match accuracy: {acc:.4f}")
else:
    print("Accuracy skipped (no predictions).")


Accuracy skipped (no predictions).


In [ ]:
# This is harsh for free-form text (almost always zero) because free-form text can vary greatly with a huge range of possible outcomes. Every response is likely to be slightly different, so the chance of an exact string match is very low.


### Part V. ROUGE metric implementation
Use `evaluate.load("rouge")` and NLTK sentence tokenizer.
Preprocess by joining sentences with newlines for better ROUGE-L.


In [7]:
import evaluate
from nltk.tokenize import sent_tokenize
from typing import List

rouge = evaluate.load('rouge')

def normalize_text(text):
    sents = sent_tokenize(text.strip())
    # Join with newline for better ROUGE-L as per common practice
    return "\n".join(sents)

def compute_rouge_score(preds: List[str], refs: List[str]):
    # TODO: normalize preds/refs; call rouge.compute
    normalized_preds = [normalize_text(p) for p in preds]
    # refs can be a list of lists if multiple references per prediction are supported
    # For this exercise, assuming a single reference per prediction, so normalize directly.
    normalized_refs = [normalize_text(r) for r in refs]

    # compute ROUGE scores, typically using stemming
    results = rouge.compute(predictions=normalized_preds, references=normalized_refs, use_stemmer=False)
    return results

# Smoke test with identical strings and empty prediction
test_preds = ["alpha beta", "", "The cat sat."]
test_refs  = ["alpha beta", "reference text", "The cat sat."]
print("ROUGE sanity check (fill function first):")
print(compute_rouge_score(test_preds, test_refs))


ROUGE sanity check (fill function first):
{'rouge1': np.float64(0.6666666666666666), 'rouge2': np.float64(0.6666666666666666), 'rougeL': np.float64(0.6666666666666666), 'rougeLsum': np.float64(0.6666666666666666)}



### Part VI. Understanding ROUGE scores
Experiments to run (describe your findings in a text cell):
- Exact match vs. empty prediction.
- Effect of stemming: e.g., "running" vs. "run".
- N-gram overlap: see how ROUGE-1 vs. ROUGE-2 change with partial overlap.
- Symmetry: swap preds/refs and compare.


### Experiment: Exact Match vs. Empty Prediction

In [8]:
print('\n--- Experimenting with Empty Predictions ---')

# Test case 1: Empty prediction vs. non-empty reference
preds_empty_vs_nonempty = ["", "hello world"]
refs_empty_vs_nonempty = ["this is a reference", "hello world"]
print("\nCase 1: Empty prediction vs. non-empty reference")
print(f"Predictions: {preds_empty_vs_nonempty}")
print(f"References: {refs_empty_vs_nonempty}")
print(f"Accuracy: {compute_accuracy(preds_empty_vs_nonempty, refs_empty_vs_nonempty):.4f}")
print("ROUGE scores:")
print(compute_rouge_score(preds_empty_vs_nonempty, refs_empty_vs_nonempty))

# Test case 2: Empty prediction vs. empty reference
preds_empty_vs_empty = ["", "test"]
refs_empty_vs_empty = ["", "test"]
print("\nCase 2: Empty prediction vs. empty reference")
print(f"Predictions: {preds_empty_vs_empty}")
print(f"References: {refs_empty_vs_empty}")
print(f"Accuracy: {compute_accuracy(preds_empty_vs_empty, refs_empty_vs_empty):.4f}")
print("ROUGE scores:")
print(compute_rouge_score(preds_empty_vs_empty, refs_empty_vs_empty))

# Test case 3: Non-empty prediction vs. empty reference
preds_nonempty_vs_empty = ["hello", "world"]
refs_nonempty_vs_empty = ["", "world"]
print("\nCase 3: Non-empty prediction vs. empty reference")
print(f"Predictions: {preds_nonempty_vs_empty}")
print(f"References: {refs_nonempty_vs_empty}")
print(f"Accuracy: {compute_accuracy(preds_nonempty_vs_empty, refs_nonempty_vs_empty):.4f}")
print("ROUGE scores:")
print(compute_rouge_score(preds_nonempty_vs_empty, refs_nonempty_vs_empty))



--- Experimenting with Empty Predictions ---

Case 1: Empty prediction vs. non-empty reference
Predictions: ['', 'hello world']
References: ['this is a reference', 'hello world']
Accuracy: 0.5000
ROUGE scores:
{'rouge1': np.float64(0.5), 'rouge2': np.float64(0.5), 'rougeL': np.float64(0.5), 'rougeLsum': np.float64(0.5)}

Case 2: Empty prediction vs. empty reference
Predictions: ['', 'test']
References: ['', 'test']
Accuracy: 1.0000
ROUGE scores:
{'rouge1': np.float64(0.5), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.5), 'rougeLsum': np.float64(0.5)}

Case 3: Non-empty prediction vs. empty reference
Predictions: ['hello', 'world']
References: ['', 'world']
Accuracy: 0.5000
ROUGE scores:
{'rouge1': np.float64(0.5), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.5), 'rougeLsum': np.float64(0.5)}


As you can observe from the output:

*   **Accuracy:** An empty prediction only matches an empty reference. If the reference is non-empty, accuracy will be 0 for that prediction.
*   **ROUGE:** If a prediction is empty, ROUGE scores (rouge1, rouge2, rougeL, rougeLsum) will generally be 0, regardless of the reference. This is because there are no common n-grams. If both prediction and reference are empty, ROUGE scores are also typically 0 or might yield NaN/division by zero errors depending on the exact implementation, as there's no content to compare.

### Experiment: Effect of Stemming

In [9]:
print('\n--- Experimenting with Stemming ---')

def compute_rouge_score_with_stemming(preds: List[str], refs: List[str]):
    normalized_preds = [normalize_text(p) for p in preds]
    normalized_refs = [normalize_text(r) for r in refs]
    # Compute ROUGE scores with stemming enabled
    results = rouge.compute(predictions=normalized_preds, references=normalized_refs, use_stemmer=True)
    return results

# Test cases to show the effect of stemming
preds_stemming = ["I am running fast.", "He quickly jumped."]
refs_stemming  = ["I run quickly.", "He jumps quickly."]

print("\nPredictions: ", preds_stemming)
print("References:  ", refs_stemming)

print("\nROUGE scores (with stemming enabled):")
print(compute_rouge_score_with_stemming(preds_stemming, refs_stemming))

print("\nROUGE scores (with stemming disabled - current default):")
print(compute_rouge_score(preds_stemming, refs_stemming))



--- Experimenting with Stemming ---

Predictions:  ['I am running fast.', 'He quickly jumped.']
References:   ['I run quickly.', 'He jumps quickly.']

ROUGE scores (with stemming enabled):
{'rouge1': np.float64(0.7857142857142858), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.6190476190476191), 'rougeLsum': np.float64(0.6190476190476191)}

ROUGE scores (with stemming disabled - current default):
{'rouge1': np.float64(0.47619047619047616), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.47619047619047616), 'rougeLsum': np.float64(0.47619047619047616)}


As you can observe, when stemming is enabled, words like 'running' and 'run' or 'jumped' and 'jumps' are reduced to their root form. This can lead to higher ROUGE scores as it increases the overlap of n-grams that are semantically similar but morphologically different. For example, 'running' and 'run' both stem to 'run', contributing to ROUGE-1 if 'run' is in the reference. Without stemming, 'running' and 'run' are considered different tokens, reducing the overlap.

### Experiment: N-gram Overlap

In [10]:
print('\n--- Experimenting with N-gram Overlap ---')

# Test case 1: High ROUGE-1, low ROUGE-2 (many unigram overlaps, few bigram overlaps)
preds_partial_overlap_1 = ["the quick brown fox jumps"]
refs_partial_overlap_1  = ["a quick red fox leaps"]
print("\nCase 1: High Unigram, Low Bigram Overlap")
print(f"Predictions: {preds_partial_overlap_1}")
print(f"References:  {refs_partial_overlap_1}")
print("ROUGE scores:")
print(compute_rouge_score(preds_partial_overlap_1, refs_partial_overlap_1))

# Test case 2: Higher ROUGE-2 (more bigram overlaps)
preds_partial_overlap_2 = ["the quick brown fox jumps over"]
refs_partial_overlap_2  = ["the quick brown dog jumps high"]
print("\nCase 2: Higher Bigram Overlap")
print(f"Predictions: {preds_partial_overlap_2}")
print(f"References:  {refs_partial_overlap_2}")
print("ROUGE scores:")
print(compute_rouge_score(preds_partial_overlap_2, refs_partial_overlap_2))

# Test case 3: Minimal overlap
preds_partial_overlap_3 = ["cat dog"]
refs_partial_overlap_3  = ["apple banana"]
print("\nCase 3: Minimal Overlap")
print(f"Predictions: {preds_partial_overlap_3}")
print(f"References:  {refs_partial_overlap_3}")
print("ROUGE scores:")
print(compute_rouge_score(preds_partial_overlap_3, refs_partial_overlap_3))


--- Experimenting with N-gram Overlap ---

Case 1: High Unigram, Low Bigram Overlap
Predictions: ['the quick brown fox jumps']
References:  ['a quick red fox leaps']
ROUGE scores:
{'rouge1': np.float64(0.4000000000000001), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.4000000000000001), 'rougeLsum': np.float64(0.4000000000000001)}

Case 2: Higher Bigram Overlap
Predictions: ['the quick brown fox jumps over']
References:  ['the quick brown dog jumps high']
ROUGE scores:
{'rouge1': np.float64(0.6666666666666666), 'rouge2': np.float64(0.4000000000000001), 'rougeL': np.float64(0.6666666666666666), 'rougeLsum': np.float64(0.6666666666666666)}

Case 3: Minimal Overlap
Predictions: ['cat dog']
References:  ['apple banana']
ROUGE scores:
{'rouge1': np.float64(0.0), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.0), 'rougeLsum': np.float64(0.0)}


From these experiments, you can observe:

*   **ROUGE-1** measures the overlap of unigrams (single words). Even if sentence structure or phrases are different, if individual words match, ROUGE-1 will be higher. In Case 1, 'quick' and 'fox' contribute to ROUGE-1.
*   **ROUGE-2** measures the overlap of bigrams (two-word sequences). For ROUGE-2 to be high, not only do individual words need to match, but they also need to appear consecutively. In Case 2, 'the quick' and 'quick brown' contribute to a higher ROUGE-2 score compared to Case 1, where 'brown fox' and 'red fox' didn't yield bigram matches.
*   If there's very little to no word overlap, as in Case 3, both ROUGE-1 and ROUGE-2 scores will be very low or zero.

### Experiment: Symmetry (Swap Predictions and References)

In [11]:
print('\n--- Experimenting with Symmetry (Swap Preds/Refs) ---')

preds_symmetry = ["The quick brown fox."]
refs_symmetry  = ["A brown fox is quick."]

print("\nOriginal (Preds -> Refs):")
print(f"Predictions: {preds_symmetry}")
print(f"References:  {refs_symmetry}")
print("ROUGE scores:")
print(compute_rouge_score(preds_symmetry, refs_symmetry))

print("\nSwapped (Refs -> Preds):")
print(f"Predictions: {refs_symmetry}")
print(f"References:  {preds_symmetry}")
print("ROUGE scores:")
print(compute_rouge_score(refs_symmetry, preds_symmetry))


--- Experimenting with Symmetry (Swap Preds/Refs) ---

Original (Preds -> Refs):
Predictions: ['The quick brown fox.']
References:  ['A brown fox is quick.']
ROUGE scores:
{'rouge1': np.float64(0.6666666666666665), 'rouge2': np.float64(0.28571428571428575), 'rougeL': np.float64(0.4444444444444445), 'rougeLsum': np.float64(0.4444444444444445)}

Swapped (Refs -> Preds):
Predictions: ['A brown fox is quick.']
References:  ['The quick brown fox.']
ROUGE scores:
{'rouge1': np.float64(0.6666666666666665), 'rouge2': np.float64(0.28571428571428575), 'rougeL': np.float64(0.4444444444444445), 'rougeLsum': np.float64(0.4444444444444445)}


From this experiment, we observe that ROUGE scores (ROUGE-1, ROUGE-2, ROUGE-L, and ROUGE-Lsum) are symmetrical. This means that swapping the predictions and references does not change the resulting scores. This property is expected, as ROUGE measures the overlap between two texts, and the calculation of overlap is not dependent on which text is designated as the prediction and which as the reference.


### Part VII. Comparing small and large models
Goals:
- Generate summaries with `t5-small`, `t5-base`, and `gpt2` (TL;DR style prompt).
- Compute ROUGE for each and store per-row scores.
- Implement `compute_rouge_per_row` to add ROUGE columns to a DataFrame.
- Implement `summarize_with_gpt2` with a TL;DR: prefix and max length guard.
Use small batches and low `max_new_tokens` to keep things snappy.


In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, gc
import pandas as pd
from typing import List

# Assuming batch_generator and compute_rouge_score are defined in previous cells
# (They are, so we'll just call them here)

def batch_generator(items: List[str], batch_size: int):
    # Re-defining here for self-contained cell, though it exists above
    for i in range(0, len(items), batch_size):
        yield items[i : i + batch_size]

def summarize_with_gpt2(texts: List[str], model_name: str = 'gpt2', batch_size: int = 2, max_new_tokens: int = 32):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # Set pad_token for GPT-2 tokenizer, which often lacks one by default
    tokenizer.pad_token = tokenizer.eos_token
    # Set padding_side to 'left' for decoder-only models like GPT-2 for correct generation
    tokenizer.padding_side = 'left'
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

    all_summaries = []
    for i, batch_texts in enumerate(batch_generator(texts, batch_size)):
        # Prepend 'TL;DR:' to prompt for summarization
        inputs = ["TL;DR: " + text for text in batch_texts]
        tokenized_inputs = tokenizer(inputs, return_tensors='pt', padding=True, truncation=True, max_length=512).to(device)

        summary_ids = model.generate(
            tokenized_inputs.input_ids,
            attention_mask=tokenized_inputs.attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=True, # GPT-2 is typically better with sampling for text generation
            temperature=0.7,
            top_k=50,
            top_p=0.95
        )

        decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True) for s in summary_ids]
        all_summaries.extend(decoded_summaries)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    return all_summaries

def compute_rouge_per_row(df: pd.DataFrame, pred_col: str, ref_col: str = 'prompt_title'):
    # Make a copy to avoid modifying the original DataFrame in place
    df_copy = df.copy()

    # `rouge` object and `normalize_text` function are assumed to be accessible globally
    # from cell 3ef0a7df, removing redundant imports/definitions.

    # Function to compute ROUGE for a single pair of prediction and reference
    def _compute_single_rouge(prediction, reference):
        # Use globally defined normalize_text
        normalized_pred = normalize_text(prediction)
        normalized_ref = normalize_text(reference)
        # compute ROUGE scores, typically using stemming
        # Using `use_stemmer=False` as per previous experiment results/discussion
        if normalized_pred == "" or normalized_ref == "": # Handle empty strings gracefully
            return {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0, 'rougeLsum': 0.0}

        # Use globally defined rouge object
        results = rouge.compute(predictions=[normalized_pred], references=[normalized_ref], use_stemmer=False)
        return results

    # Apply the ROUGE computation row-wise
    rouge_scores = df_copy.apply(lambda row: _compute_single_rouge(row[pred_col], row[ref_col]), axis=1)

    # Expand the dictionary results into new columns in the DataFrame
    df_copy[f'{pred_col}_rouge1'] = [s['rouge1'] for s in rouge_scores]
    df_copy[f'{pred_col}_rouge2'] = [s['rouge2'] for s in rouge_scores]
    df_copy[f'{pred_col}_rougeL'] = [s['rougeL'] for s in rouge_scores]
    df_copy[f'{pred_col}_rougeLsum'] = [s['rougeLsum'] for s in rouge_scores]

    return df_copy

RUN_COMPARE = True # Set to True to enable the comparison

if RUN_COMPARE:
    print("Starting model comparison...")
    results_df = train_df.copy() # Start with the training data for comparison

    # --- Generate T5-small summaries ---
    # Check if t5-small summaries already exist (e.g., from RUN_T5 = True earlier)
    if 'train_summaries_t5' not in locals():
        print("Generating T5-small summaries...")
        train_summaries_t5 = summarize_with_t5(train_df['prompt_text'].tolist(), model_name='t5-small', batch_size=2)
    else:
        print("Using existing T5-small summaries.")
    results_df['t5_small_summary'] = train_summaries_t5

    # --- Generate T5-base summaries ---
    print("Generating T5-base summaries...")
    t5_base_summaries = summarize_with_t5(train_df['prompt_text'].tolist(), model_name='t5-base', batch_size=2)
    results_df['t5_base_summary'] = t5_base_summaries

    # --- Generate GPT-2 summaries ---
    print("Generating GPT-2 summaries...")
    gpt2_summaries = summarize_with_gpt2(train_df['prompt_text'].tolist(), model_name='gpt2', batch_size=2)
    results_df['gpt2_summary'] = gpt2_summaries

    # --- Compute per-row ROUGE scores ---
    print("Computing per-row ROUGE scores...")
    results_df = compute_rouge_per_row(results_df, 't5_small_summary')
    results_df = compute_rouge_per_row(results_df, 't5_base_summary')
    results_df = compute_rouge_per_row(results_df, 'gpt2_summary')

    print("Model comparison complete. Displaying results head:")
    display(results_df.head())
else:
    print("Skipping model comparison. Set RUN_COMPARE=True to execute.")

Starting model comparison...
Generating T5-small summaries...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end gene

Computing per-row ROUGE scores...
Model comparison complete. Displaying results head:


,prompt_text,prompt_title,t5_small_summary,t5_base_summary,gpt2_summary,t5_small_summary_rouge1,t5_small_summary_rouge2,t5_small_summary_rougeL,t5_small_summary_rougeLsum,t5_base_summary_rouge1,t5_base_summary_rouge2,t5_base_summary_rougeL,t5_base_summary_rougeLsum,gpt2_summary_rouge1,gpt2_summary_rouge2,gpt2_summary_rougeL,gpt2_summary_rougeLsum
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...,championship leader Lewis Hamilton spun out of...,championship leader Lewis Hamilton spins out o...,"TL;DR: SHANGHAI, China -- Championship leader ...",0.349206,0.163934,0.253968,0.253968,0.507937,0.295082,0.507937,0.507937,0.142276,0.081633,0.101626,0.138211
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...,china suspends exports of the toys that contai...,china suspends exports of the Aqua Dots toys c...,TL;DR: (CNN) -- China has suspended exports of...,0.144928,0.000000,0.115942,0.115942,0.202899,0.029851,0.144928,0.173913,0.189873,0.122881,0.147679,0.181435
2,(CNN) -- The company was founded in 1985 by se...,The company has become a huge name in communic...,the company was founded in 1985 by seven commu...,the company was founded in 1985 by seven commu...,TL;DR: (CNN) -- The company was founded in 198...,0.235294,0.060606,0.235294,0.235294,0.338462,0.095238,0.246154,0.338462,0.261603,0.093617,0.194093,0.244726
3,"ISLAMABAD, Pakistan (CNN) -- Hours after decla...",NEW: President Musharraf orders troops to take...,president pervez Musharraf orders troops to ta...,new: police arrest acting president of ex-Prim...,"TL;DR: ISLAMABAD, Pakistan (CNN) -- Hours afte...",0.412698,0.229508,0.380952,0.412698,0.285714,0.000000,0.222222,0.253968,0.143791,0.078775,0.117647,0.139434
4,"QUEBEC, Canada -- Third seed Julia Vakulenko w...",Julia Vakulenko has reached her first final on...,third seed Julia Vakulenko will face comeback ...,third seed Julia vakulenko will face former wo...,"TL;DR: QUEBEC, Canada -- Third seed Julia Vaku...",0.448276,0.178571,0.275862,0.448276,0.533333,0.241379,0.333333,0.533333,0.151899,0.084746,0.109705,0.147679



### Part VIII. Comparing all models
Implement:
- `compare_models` to aggregate average ROUGE across models.
- `compare_models_summaries` to show side-by-side summaries.
Present the tables and discuss which model wins and why.


In [15]:

import pandas as pd

def compare_models(rouge_dict):
    # Take {model_name: rouge_scores_dict} -> DataFrame with averages
    df_avg_rouge = pd.DataFrame.from_dict(rouge_dict, orient='index')
    df_avg_rouge.index.name = 'Model'
    # Sort columns for better readability (e.g., rouge1, rouge2, rougeL, rougeLsum)
    ordered_cols = ['rouge1', 'rouge2', 'rougeL', 'rougeLsum']
    return df_avg_rouge[ordered_cols]

def compare_models_summaries(df: pd.DataFrame, pred_cols: list):
    # Subset columns for side-by-side viewing
    display_cols = ['prompt_text', 'prompt_title'] + pred_cols
    return df[display_cols]

# --- Execute comparison and discussion ---

if 'results_df' in locals():
    print("\n--- Average ROUGE Scores Across Models ---")

    model_rouge_averages = {}
    model_names = ['t5_small', 't5_base', 'gpt2']

    for model_name in model_names:
        rouge_metrics = ['rouge1', 'rouge2', 'rougeL', 'rougeLsum']
        avg_scores = {
            metric: results_df[f'{model_name}_summary_{metric}'].mean()
            for metric in rouge_metrics
        }
        model_rouge_averages[model_name] = avg_scores

    avg_rouge_df = compare_models(model_rouge_averages)
    display(avg_rouge_df)

    print("\n--- Side-by-Side Summaries (First 5 examples) ---")
    prediction_columns = [
        't5_small_summary',
        't5_base_summary',
        'gpt2_summary'
    ]
    side_by_side_df = compare_models_summaries(results_df.head(5), prediction_columns)
    display(side_by_side_df)

else:
    print("Please ensure 'results_df' is available by running previous cells, especially the model comparison section.")



--- Average ROUGE Scores Across Models ---


,rouge1,rouge2,rougeL,rougeLsum
Model,,,,
t5_small,0.262045,0.096630,0.196532,0.237476
t5_base,0.294366,0.127378,0.224792,0.274233
gpt2,0.145451,0.066417,0.102123,0.138457



--- Side-by-Side Summaries (First 5 examples) ---


,prompt_text,prompt_title,t5_small_summary,t5_base_summary,gpt2_summary
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...,championship leader Lewis Hamilton spun out of...,championship leader Lewis Hamilton spins out o...,"TL;DR: SHANGHAI, China -- Championship leader ..."
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...,china suspends exports of the toys that contai...,china suspends exports of the Aqua Dots toys c...,TL;DR: (CNN) -- China has suspended exports of...
2,(CNN) -- The company was founded in 1985 by se...,The company has become a huge name in communic...,the company was founded in 1985 by seven commu...,the company was founded in 1985 by seven commu...,TL;DR: (CNN) -- The company was founded in 198...
3,"ISLAMABAD, Pakistan (CNN) -- Hours after decla...",NEW: President Musharraf orders troops to take...,president pervez Musharraf orders troops to ta...,new: police arrest acting president of ex-Prim...,"TL;DR: ISLAMABAD, Pakistan (CNN) -- Hours afte..."
4,"QUEBEC, Canada -- Third seed Julia Vakulenko w...",Julia Vakulenko has reached her first final on...,third seed Julia Vakulenko will face comeback ...,third seed Julia vakulenko will face former wo...,"TL;DR: QUEBEC, Canada -- Third seed Julia Vaku..."


### Discussion: Which model wins and why?

Based on the average ROUGE scores, the `t5_base` model generally outperforms `t5_small` and `gpt2` across all ROUGE metrics (rouge1, rouge2, rougeL, and rougeLsum).

*   **ROUGE Scores:** The higher scores for `t5_base` indicate a better overlap in unigrams (rouge1), bigrams (rouge2), and overall sentence structure/main points (rougeL, rougeLsum) compared to the reference summaries. This is expected, as `t5-base` is a larger model than `t5-small` and has likely learned more robust summarization patterns.

*   **Qualitative Quality:** Observing the side-by-side summaries, `t5_base` often produces more coherent, grammatically correct, and semantically richer summaries than `t5_small`. While `gpt2`'s summaries are generative, they can sometimes be less concise or focused compared to the T5 models, which are specifically designed and often fine-tuned for summarization. GPT-2 might occasionally introduce irrelevant details or lack the compression capabilities of summarization-focused models.

*   **Model Specialization:** T5 models are encoder-decoder architectures often fine-tuned for summarization as a seq2seq task, making them highly effective. GPT-2, a decoder-only model, excels at open-ended text generation but may require more explicit prompting or further fine-tuning to achieve optimal summarization performance comparable to T5-specific models.

In conclusion, **`t5_base` is the 'winner' in this comparison** due to its superior performance across ROUGE metrics and generally higher qualitative summary generation. This highlights that model size and task-specific architecture/training are crucial factors for achieving high-quality summarization.


## Wrap-up
- Which metrics felt most informative? Why?
- How did model size impact ROUGE and qualitative quality?
- Where did accuracy break down as a metric?
- How would you extend this to human eval or adversarial probes?
Write a short reflection here.


ROUGE scores were most informative, much more than simple exact-match accuracy, because accuracy was almost always zero for free-form text, as exact string matches are rarely achieved with generative models; ROUGE scores, meanwhile, which measure the overlap of n-grams, allow for a nuanced evaluation, including word overlap, phrase similarity, and order of main points (for ROUGE-1, ROUGE-2, and ROUGE-L).


Based on the comparison, model size significantly impacted both ROUGE scores and qualitative quality. The t5_base model, being larger than t5_small, consistently achieved higher ROUGE scores across all metrics (rouge1, rouge2, rougeL, rougeLsum). Qualitatively, t5_base produced more coherent, grammatically correct, and semantically richer summaries compared to t5_small. This suggests that larger models, especially those fine-tuned for summarization like T5, generally learn more robust summarization patterns and yield better results.


Accuracy broke down as a metric primarily because it relies on an exact string match between the generated summary and the reference summary. For free-form text generation tasks like summarization, generative models rarely produce an exact match, even if the content is semantically very similar. This often leads to an accuracy score of zero, making it uninformative for evaluating the quality of the generated text. ROUGE scores, by contrast, offer a more nuanced evaluation by measuring n-gram overlap, which accounts for partial matches and semantic similarity more effectively.


To extend this evaluation to human assessment and adversarial probes, you could consider the following:

Human Evaluation:

    Methodology: You would engage human annotators to evaluate the generated summaries based on criteria that ROUGE scores don't fully capture, such as fluency, coherence, factual consistency, and overall readability. This could involve:
        Rating scales: Humans rate summaries on a Likert scale (e.g., 1-5) for each quality attribute.
        Ranking: Present multiple summaries for the same source text and ask annotators to rank them from best to worst.
        Error analysis: Annotators identify specific types of errors like hallucinations (generating facts not in the source), factual inaccuracies, or grammatical mistakes.
    Benefits: Provides a more nuanced understanding of summary quality, reflecting user experience.
    Challenges: It's expensive, time-consuming, and requires careful design to ensure inter-annotator agreement and minimize subjectivity.

Adversarial Probes:

    Methodology: This involves intentionally creating challenging inputs to test the model's robustness and identify its limitations. For summarization, this might include:
        Factuality probes: Introduce subtle factual errors or contradictions into the source text and observe if the model's summary propagates these errors (hallucination) or correctly handles them.
        Redundancy/Distractor injection: Add irrelevant sentences or repetitive information to the source article to see if the model can still generate a concise and focused summary, ignoring the noise.
        Bias detection: Craft prompts that explore potential biases in how the model summarizes information related to different demographics or sensitive topics.
        Abstractive quality: Test the model's ability to paraphrase and synthesize information rather than just extracting sentences, by providing texts that require deeper understanding and rephrasing.
    Benefits: Uncovers blind spots and vulnerabilities that traditional metrics might miss, leading to more robust models.
    Challenges: Requires creativity and domain expertise to design effective and insightful adversarial examples.

